In [660]:
import pandas as pd
import numpy as np
import re 
import geopandas as gpd

In [ ]:
# Loading datasets

zhvi = pd.read_csv(r"..\..\Data\County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")
fema = pd.read_excel(r"..\..\Data\fema_2000_2025.xlsx")
state_code =  pd.read_excel(r"..\..\Data\state_code.xlsx")
fema_county = pd.read_csv(r"..\..\Data\DisasterDeclarationsSummaries.csv")

In [662]:
fema_county.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69767 entries, 0 to 69766
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   femaDeclarationString     69767 non-null  object
 1   disasterNumber            69767 non-null  int64 
 2   state                     69767 non-null  object
 3   declarationType           69767 non-null  object
 4   declarationDate           69767 non-null  object
 5   fyDeclared                69767 non-null  int64 
 6   incidentType              69767 non-null  object
 7   declarationTitle          69767 non-null  object
 8   ihProgramDeclared         69767 non-null  int64 
 9   iaProgramDeclared         69767 non-null  int64 
 10  paProgramDeclared         69767 non-null  int64 
 11  hmProgramDeclared         69767 non-null  int64 
 12  incidentBeginDate         69767 non-null  object
 13  incidentEndDate           69232 non-null  object
 14  disasterCloseoutDate  

In [663]:
fema_county.iloc[:,:10]

,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,iaProgramDeclared
0,FM-5529-OR,5529,OR,FM,2024-08-09T00:00:00.000Z,2024,Fire,LEE FALLS FIRE,0,0
1,FM-5528-OR,5528,OR,FM,2024-08-06T00:00:00.000Z,2024,Fire,ELK LANE FIRE,0,0
2,FM-5527-OR,5527,OR,FM,2024-08-02T00:00:00.000Z,2024,Fire,MILE MARKER 132 FIRE,0,0
3,DR-4312-CA,4312,CA,DR,2017-05-02T00:00:00.000Z,2017,Severe Storm,FLOODING,0,0
4,DR-4251-AL,4251,AL,DR,2016-01-21T00:00:00.000Z,2016,Severe Storm,"SEVERE STORMS, TORNADOES, STRAIGHT-LINE WINDS,...",0,0
...,...,...,...,...,...,...,...,...,...,...
69762,DR-9-TX,9,TX,DR,1953-06-19T00:00:00.000Z,1953,Flood,FLOOD,0,1
69763,DR-8-IA,8,IA,DR,1953-06-11T00:00:00.000Z,1953,Flood,FLOOD,0,1
69764,DR-7-MA,7,MA,DR,1953-06-11T00:00:00.000Z,1953,Tornado,TORNADO,0,1
69765,DR-2-TX,2,TX,DR,1953-05-15T00:00:00.000Z,1953,Tornado,TORNADO & HEAVY RAINFALL,0,1


In [664]:
fema_county.fyDeclared.value_counts()

fyDeclared
2020    9490
2005    4692
2011    2684
2008    2430
2021    2156
        ... 
1960      13
1961      11
1953      10
1959       8
1958       5
Name: count, Length: 74, dtype: int64

### I. Cleaning FEMA’s Disaster Declarations record

In [ ]:
# Filter dataset - hurricanes classified as major disasters and taking out other US territories (islands)

exclude = ['AK','HI','PR','GU','AS','MP','VI','FM','MH','PW']

fema_county_copy = fema_county[~fema_county.state.isin(exclude)].copy()
fema_county_copy = (fema_county_copy[
    (fema_county_copy.declarationType=='DR')  #  major disasters 
    & (fema_county_copy.incidentType=="Hurricane") # filter out for hurricanes
    & (fema_county_copy.fyDeclared>=2000)]
)

# selecting columns 

fema_county_filter = fema_county_copy[['disasterNumber',
                                       'state',
                                       'designatedArea',
                                       'fyDeclared',
                                       'incidentBeginDate',
                                       'incidentEndDate',           
                                       'declarationTitle',
                                       'fipsStateCode',
                                       'fipsCountyCode']]

# Year when the hurricane hits county

def reso_info(x):
    
    regex_search = re.compile('^([0-9]+)-.*') # withdraw firs 4 digits (year of disaster)
    match = regex_search.search(x)
    
    return match.group(1)

fema_county_filter['forward_year'] = fema_county_filter['incidentBeginDate'].apply(lambda x: reso_info(x)).astype(int) + 1

# Hurriccane name in Title format

fema_county_filter['declarationTitle'] = fema_county_filter['declarationTitle'].str.title()
fema_county_filter['dhurricane'] = 1

# Taking out ares with fipsCountyCode equals 0, refering to reservation o pretected areas 

fema_county_filter = fema_county_filter[fema_county_filter.fipsCountyCode!=0]

C:\Users\rmend\AppData\Local\Temp\ipykernel_24400\788554783.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fema_county_filter['forward_year'] = fema_county_filter['incidentBeginDate'].apply(lambda x: reso_info(x)).astype(int) + 1
C:\Users\rmend\AppData\Local\Temp\ipykernel_24400\788554783.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fema_county_filter['declarationTitle'] = fema_county_filter['declarationTitle'].str.title()
C:\Users\rmend\AppData\Local\Temp\ipykernel_24400\788554783.py:38: Sett

In [666]:
# Counting the number of hurricanes that strike more than once per year in a county

fema_county_filter['count_dup'] = (
    fema_county_filter.groupby(['state','fipsStateCode',	'fipsCountyCode',	'forward_year']).cumcount() + 1
)

In [667]:
fema_county_filter_1 = fema_county_filter[fema_county_filter.count_dup==1] # one hurricane ina county-year
fema_county_filter_2 = fema_county_filter[fema_county_filter.count_dup==2] # two hurricane ina county-year
fema_county_filter_3 = fema_county_filter[fema_county_filter.count_dup==3] # three hurricane ina county-year

In [668]:
# Reshape in wide format for hurricanes name

fema_county_filter_all = pd.merge(
    fema_county_filter_1,
    fema_county_filter_2[['state','fipsStateCode',	'fipsCountyCode',	'forward_year', 'declarationTitle']],
    how = 'left',
    on = ['state','fipsStateCode',	'fipsCountyCode',	'forward_year'],
    validate = "1:1"
).merge(
    fema_county_filter_3[['state','fipsStateCode',	'fipsCountyCode',	'forward_year', 'declarationTitle']],
    how = 'left',
    on = ['state','fipsStateCode',	'fipsCountyCode',	'forward_year'],
    validate = "1:1"
)

In [669]:
fema_county_filter_all.rename(
    columns={'declarationTitle_x':'declarationTitle1',
             'declarationTitle_y':'declarationTitle2',
             'declarationTitle':'declarationTitle3'
             },
    inplace=True
)

In [670]:
cols = ['declarationTitle1', 'declarationTitle2', 'declarationTitle3']

# Join hurricane names for years where monre than one hurricane strikes in a  county

fema_county_filter_all['hurricane_name'] = (
    fema_county_filter_all[cols]
    .apply(
        lambda row: ' ,'.join(
            row.dropna()
            .str.replace("Hurricane","",regex=False)
            .str.strip()
        ), axis =1
    )
)

fema_county_filter_all.drop(columns=['declarationTitle1',
                                     'declarationTitle2',
                                     'declarationTitle3',
                                     'count_dup'],
                            inplace=True)

In [671]:
fema_county_filter_all['date2'] = pd.to_datetime(fema_county_filter_all['incidentEndDate'],
                                      format='mixed')

fema_county_filter_all['date1'] = fema_county_filter_all['date2'].dt.year.astype('Int64').astype(str) + 'm' + fema_county_filter_all['date2'].dt.month.astype('Int64').astype(str)

fema_county_filter_all.drop(columns=['incidentEndDate','date2'],
                            inplace=True)

# Expanding 12 additional months after the hurricane ends

fema_county_filter_all['month_offset'] = [list(range(12))] * len(fema_county_filter_all)


# stacking additional months

df_expanded = fema_county_filter_all.explode('month_offset')

df_expanded['date1'] = pd.to_datetime(df_expanded['date1'], format='%Ym%m')

df_expanded['date'] = df_expanded.apply(
    lambda x: x['date1'] + pd.DateOffset(months=x['month_offset']),
    axis=1
)

df_expanded['date'] = df_expanded['date'].dt.year.astype(str) + 'm' + df_expanded['date'].dt.month.astype(str)

df_expanded['date_format'] = pd.to_datetime(
    df_expanded['date'].str.replace('m', '-'),
    format='%Y-%m'
)

# droping auxiliar date column 

del df_expanded['date1'], df_expanded['month_offset']

In [672]:
df_expanded

,disasterNumber,state,designatedArea,fyDeclared,incidentBeginDate,fipsStateCode,fipsCountyCode,forward_year,dhurricane,hurricane_name,date,date_format
0,4798,TX,Anderson (County),2024,2024-07-05T00:00:00.000Z,48,1,2025,1,Beryl,2024m7,2024-07-01
0,4798,TX,Anderson (County),2024,2024-07-05T00:00:00.000Z,48,1,2025,1,Beryl,2024m8,2024-08-01
0,4798,TX,Anderson (County),2024,2024-07-05T00:00:00.000Z,48,1,2025,1,Beryl,2024m9,2024-09-01
0,4798,TX,Anderson (County),2024,2024-07-05T00:00:00.000Z,48,1,2025,1,Beryl,2024m10,2024-10-01
0,4798,TX,Anderson (County),2024,2024-07-05T00:00:00.000Z,48,1,2025,1,Beryl,2024m11,2024-11-01
...,...,...,...,...,...,...,...,...,...,...,...,...
3638,1308,ME,Somerset (County),2000,1999-09-16T00:00:00.000Z,23,25,2000,1,Floyd Major Disaster Declarations,2000m4,2000-04-01
3638,1308,ME,Somerset (County),2000,1999-09-16T00:00:00.000Z,23,25,2000,1,Floyd Major Disaster Declarations,2000m5,2000-05-01
3638,1308,ME,Somerset (County),2000,1999-09-16T00:00:00.000Z,23,25,2000,1,Floyd Major Disaster Declarations,2000m6,2000-06-01
3638,1308,ME,Somerset (County),2000,1999-09-16T00:00:00.000Z,23,25,2000,1,Floyd Major Disaster Declarations,2000m7,2000-07-01


In [673]:
df_expanded.sort_values(['fipsStateCode','fipsCountyCode','hurricane_name','date_format'],
                        inplace=True)

df_expanded = df_expanded.drop_duplicates(
    subset=['fipsStateCode','fipsCountyCode','date'],
    keep='first'
)

In [674]:
df_expanded.sort_values(['fipsStateCode','fipsCountyCode','hurricane_name','date_format'],
                        inplace=True)

C:\Users\rmend\AppData\Local\Temp\ipykernel_24400\2952861778.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_expanded.sort_values(['fipsStateCode','fipsCountyCode','hurricane_name','date_format'],


### II. Cleaning Zillow’s county-level Home Value Index dataset

In [676]:
# Droppping Alaska and Hawai  

zhvi_copy = (zhvi[~zhvi.StateName.isin(['AK',
                                       'HI'])]
             .copy()
             .drop(columns=['SizeRank','RegionType','StateName','Metro'])            
)

# reshape a long format

zhvi_long = (zhvi_copy.melt(
    id_vars=['RegionID','RegionName','State','StateCodeFIPS','MunicipalCodeFIPS'],
    var_name='fulldate',
    value_name='zhvi_value'
)
.sort_values(['RegionID','fulldate'])
.reset_index(drop=True)
)

# Full date format
zhvi_long['fulldate'] = pd.to_datetime(zhvi_long['fulldate'])

# Date in year-month
zhvi_long['date'] = zhvi_long['fulldate'].dt.year.astype(str) + 'm' + zhvi_long['fulldate'].dt.month.astype(str)

# Year
zhvi_long['year'] = zhvi_long['fulldate'].dt.year

In [677]:
zhvi_long

,RegionID,RegionName,State,StateCodeFIPS,MunicipalCodeFIPS,fulldate,zhvi_value,date,year
0,66,Ada County,ID,16,1,2000-01-31,NaN,2000m1,2000
1,66,Ada County,ID,16,1,2000-02-29,NaN,2000m2,2000
2,66,Ada County,ID,16,1,2000-03-31,NaN,2000m3,2000
3,66,Ada County,ID,16,1,2000-04-30,NaN,2000m4,2000
4,66,Ada County,ID,16,1,2000-05-31,NaN,2000m5,2000
...,...,...,...,...,...,...,...,...,...
959265,3291,Colonial Heights City,VA,51,570,2025-10-31,275358.335483,2025m10,2025
959266,3291,Colonial Heights City,VA,51,570,2025-11-30,276042.106349,2025m11,2025
959267,3291,Colonial Heights City,VA,51,570,2025-12-31,276634.365192,2025m12,2025
959268,3291,Colonial Heights City,VA,51,570,2026-01-31,276545.436016,2026m1,2026


In [678]:
# Merge Zillow’s county-level Home Value Index and Zillow’s county-level Home Value Index

dataset_merge = pd.merge(
    zhvi_long,
    df_expanded,
    how = 'left',
    left_on=['State','StateCodeFIPS','MunicipalCodeFIPS','date'],
    right_on=['state','fipsStateCode',	'fipsCountyCode','date'],
    validate='1:1'
)

In [679]:
# Replace nan in dummy hurricane per county and year

dataset_merge['dhurricane'] = dataset_merge['dhurricane'].fillna(0)

# Indicator of counties affected by at least one hurricane, spanning 2020-2026

dataset_merge['indicator'] = dataset_merge.groupby(['StateCodeFIPS','MunicipalCodeFIPS'])['dhurricane'].transform(max)

dataset_merge = dataset_merge[dataset_merge.indicator==1]

C:\Users\rmend\AppData\Local\Temp\ipykernel_24400\4091586272.py:7: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  dataset_merge['indicator'] = dataset_merge.groupby(['StateCodeFIPS','MunicipalCodeFIPS'])['dhurricane'].transform(max)


In [680]:
# State and county in the right format

dataset_merge['StateCodeFIPS'] = dataset_merge['StateCodeFIPS'].astype(str).str.zfill(2)
dataset_merge['MunicipalCodeFIPS'] = dataset_merge['MunicipalCodeFIPS'].astype(str).str.zfill(3)

# Creating Country ID similar to geoJson County USA

dataset_merge['GEO_ID'] = ("0500000US" +
    dataset_merge['StateCodeFIPS'] +
    dataset_merge['MunicipalCodeFIPS']
)

In [681]:
state_map = {
    "TX": "Texas",
    "GA": "Georgia",
    "VA": "Virginia",
    "NC": "North Carolina",
    "MS": "Mississippi",
    "AL": "Alabama",
    "PA": "Pennsylvania",
    "FL": "Florida",
    "LA": "Louisiana",
    "SC": "South Carolina",
    "NY": "New York",
    "WV": "West Virginia",
    "MD": "Maryland",
    "NJ": "New Jersey",
    "AR": "Arkansas",
    "VT": "Vermont",
    "MA": "Massachusetts",
    "NH": "New Hampshire",
    "ME": "Maine",
    "CT": "Connecticut",
    "CA": "California",
    "RI": "Rhode Island",
    "DE": "Delaware",
    "OH": "Ohio",
    "DC": "District of Columbia"
}


# Clean State and County names 

dataset_merge["State"] = dataset_merge["State"].map(state_map)
dataset_merge['RegionName'] = dataset_merge['RegionName'].str.replace('County', '').str.strip()

In [682]:
dataset_merge

,RegionID,RegionName,State,StateCodeFIPS,MunicipalCodeFIPS,fulldate,zhvi_value,date,year,disasterNumber,...,fyDeclared,incidentBeginDate,fipsStateCode,fipsCountyCode,forward_year,dhurricane,hurricane_name,date_format,indicator,GEO_ID
314,67,Bay,Florida,12,005,2000-01-31,111749.313703,2000m1,2000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US12005
315,67,Bay,Florida,12,005,2000-02-29,111981.274100,2000m2,2000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US12005
316,67,Bay,Florida,12,005,2000-03-31,112202.222656,2000m3,2000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US12005
317,67,Bay,Florida,12,005,2000-04-30,112652.862673,2000m4,2000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US12005
318,67,Bay,Florida,12,005,2000-05-31,113232.573717,2000m5,2000,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US12005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959265,3291,Colonial Heights City,Virginia,51,570,2025-10-31,275358.335483,2025m10,2025,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US51570
959266,3291,Colonial Heights City,Virginia,51,570,2025-11-30,276042.106349,2025m11,2025,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US51570
959267,3291,Colonial Heights City,Virginia,51,570,2025-12-31,276634.365192,2025m12,2025,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US51570
959268,3291,Colonial Heights City,Virginia,51,570,2026-01-31,276545.436016,2026m1,2026,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaT,1.0,0500000US51570


In [ ]:
dataset_merge.to_csv(r"..\..\Data\zhvi_hurricane_dataset.csv",
                     index=False)

# Index false to not create index column in exported csv file